## ML модели для классификатора пресс релизов ##

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.svm import SVC 
import seaborn as sns
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.preprocessing import MinMaxScaler, StandardScaler
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, mean_absolute_percentage_error as MAPE

df = pd.read_csv('https://raw.github.com/HSEAi2025Team-7/keyRateForecast/dev/ML/data_ml.csv')

In [128]:
X = df[['decision_text', 'key_rate', 'key_rate_prev', 'inflation','inflation_lag_2', 
             'inflation_gap', 'usd_rate', 'text_len_tokens', 'hike_words_ratio', 'cut_words_ratio',
             'hold_words_ratio', 'hike_minus_cut_ratio']]
y = df['decision_future']

In [129]:
print(f'Размер матрицы: {X.shape}')

Размер матрицы: (54, 12)


**Описание датасета**
**decision_text** - категориальный признак, решение по ставке исходя из текущего пресс-релиза: -1 (снижение)/0 (неизменна)/1 (повышение) <br>
**key_rate** - ключевая ставка на данный момент <br>
**key_rate_prev** - ключевая ставка предыдущего пресс-релиза <br>
**inflation** - инфляция на текущий месяц <br>
**inflation_lag_2** - инфляция с лагом два (взято, так как большая корреляция) <br>
**inflation_gap** - разница между инфляций текущей и целевой <br>
**usd_rate** - курс доллара на текущую дату пресс-релиза <br>
**text_len_tokens** - количество токенов в тексте <br>
**hike_words_ratio** - доля слов, которые чаще встречаются перед повышением ставки <br>
**cut_words_ratio** - доля слов, которые чаще встречаются перед понижением ставки <br>
**hold_words_ratio** - доля слов, которые чаще встречаются при сохраненнии ставки<br>
**hike_minus_cut_ratio** - контраст между повышением и понижением <br>
**decision_future** - решение по ставке в будущем -1/0/1 <br>

Для работы выбраны ключевые метрики F1 и Accuracy так как в пресс релизах ,  классы сбалансированы и ошибки разных типов для нас равноценны, основную метрику мы выбрали accuracy для удобной интерпретации результата, где показан процент правильных ответов.
Учитывая что accuracy показывает только точность предсказания, а модель может предсказывать преобладающий класс то и точность будет достаточно высокой, поэтому для понимания что модель не пропускает классы берём F1 как дополнительную метрику, f1 учитывает и точность предсказания и полноту.

# Модели SVC и KNN

In [130]:
# разделяем датасет на трейн и тест
Xtrain, Xtest, ytrain, ytest = train_test_split(X, y, test_size=0.25, random_state=42)

In [131]:
# выделяем колонки для стандартизации
stand_col = ['key_rate', 'key_rate_prev', 'inflation', 'inflation_lag_2', 
             'inflation_gap', 'usd_rate', 'text_len_tokens']

# делаем объект масштабирования
scaler = StandardScaler()

# делаем копии, чтобы не трогать текущие данные
Xtrain_scaled = Xtrain.copy()
Xtest_scaled = Xtest.copy()
# считаем параметры стандартизации на трейне и стандартизируем признаки
Xtrain_scaled[stand_col] = scaler.fit_transform(Xtrain[stand_col])
# стандартизируем признаки для тест
Xtest_scaled[stand_col] = scaler.transform(Xtest[stand_col])

In [132]:
# задаем линейную модель
model = SVC(kernel = 'linear')

# обучаем на трейне
model.fit(Xtrain_scaled, ytrain)

# делаем предсказание на тесте
pred = model.predict(Xtest_scaled)

# считаем метрики
acc_svc = accuracy_score(ytest, pred)
f1_svc = f1_score(ytest, pred, average='weighted')

print("Accuracy:", acc_svc)
print("F1 weighted:", f1_svc)

Accuracy: 0.5
F1 weighted: 0.5111111111111111


Поберем гиперпараметры для модели а также сделаем кросс-валидацию для улучшения ее качества

In [ ]:
# создание модели
svc = SVC()

# настриваем силу регуляризации, тип ядра, способ выбора y
param_grid_svc = {
    'C': [0.1, 1, 10, 100],
    'kernel': ['linear', 'rbf'],
    'gamma': ['scale', 'auto']
}

# настриваем перебор заданных параметров и задаем кросс-валидацию на 5 фолдов
grid_svc = GridSearchCV(
    estimator=svc,
    param_grid=param_grid_svc,
    scoring='f1_macro',
    cv=5
)

# обучаем данные
grid_svc.fit(Xtrain_scaled, ytrain)

# выводим лучшие параметры и метрику
print("Лучшие параметры:", grid_svc.best_params_)
print("Лучшая F1:", grid_svc.best_score_)

# по лучшей модели проверяем тест
best_svc = grid_svc.best_estimator_
y_pred_svc = best_svc.predict(Xtest_scaled)

acc_svc_gs = accuracy_score(ytest, y_pred_svc)
f1_svc_gs = f1_score(ytest, y_pred_svc, average='macro')

# выводим метрики качества
print("Accuracy:", acc_svc_gs)
print("F1:", f1_svc_gs)

Лучшие параметры: {'C': 10, 'gamma': 'scale', 'kernel': 'rbf'}
Лучшая F1: 0.7680952380952382
Accuracy: 0.7142857142857143
F1: 0.6944444444444445


Модель дала средние результаты при стандартом применении.
Однако улучшились показатели метрик качества до 0.71 - accuracy и до 0,69 - f1 за счет применения grid search.

In [113]:
# модель KNN
knn = KNeighborsClassifier()

# параметры для подбора
param_grid = {
    'n_neighbors': range(1, 21),  # количество соседей
    'weights': ['uniform', 'distance'],  # равные веса или взвешенные по расстоянию
    'metric': ['euclidean', 'manhattan']  # тип расстояния
}

# GridSearch с 5-кратной кросс-валидацией и F1 (weighted) как метрика
grid_search = GridSearchCV(knn, param_grid, cv=5, scoring='f1_weighted')

# обучаем модель
grid_search.fit(Xtrain_scaled, ytrain)

# выводим лучшие параметры
print("Лучшие параметры:", grid_search.best_params_)

# выбираем лучшую модель
best_knn = grid_search.best_estimator_

# делаем предсказание на тесте
y_pred = best_knn.predict(Xtest_scaled)

acc_knn = accuracy_score(ytest, y_pred)
f1_knn = f1_score(ytest, y_pred, average='weighted')

# выводим метрики
print("Accuracy:", acc_knn)
print("F1 weighted:", f1_knn)


Лучшие параметры: {'metric': 'manhattan', 'n_neighbors': 3, 'weights': 'uniform'}
Accuracy: 0.6428571428571429
F1 weighted: 0.6578849721706864


показатели выше чем у модели SVM, но также стоит попробовать подобрать гиперпараметры для улучшения ее качества

In [ ]:
from sklearn.model_selection import GridSearchCV

knn = KNeighborsClassifier()

param_grid_knn = {
    'n_neighbors': [3, 5, 7, 9, 11, 13, 15, 17],
    'weights': ['uniform', 'distance']
}

grid_knn = GridSearchCV(
    estimator=knn,
    param_grid=param_grid_knn,
    scoring='f1_macro',
    cv=5
)

grid_knn.fit(Xtrain_scaled, ytrain)

print("KNN лучшие параметры:", grid_knn.best_params_)
print("KNN лучшая F1:", grid_knn.best_score_)

# оценка на тесте
best_knn = grid_knn.best_estimator_
y_pred_knn = best_knn.predict(Xtest_scaled)

acc_knn_gs = accuracy_score(ytest, y_pred_knn)
f1_knn_gs = f1_score(ytest, y_pred_knn, average='macro')

print("KNN test Accuracy:", acc_knn_gs)
print("KNN test F1:", f1_knn_gs)


KNN лучшие параметры: {'n_neighbors': 3, 'weights': 'uniform'}
KNN лучшая F1: 0.7496296296296295
KNN test Accuracy: 0.5714285714285714
KNN test F1: 0.48484848484848486


Модель работает лучше всего , когда использует три ближайших пресс-релиза, все пресс-релизы имеют одинаковый вес. На обучающих данных модель получила F1: 0.75, однако на тестовой выборке наоборот метрики снизились, что говорит о переобучении модели. <br>
В связи с этим попробуем поработать с данными и улучшить их для повышения качества работы модели. Посмотрим на корреляцию данных с таргетом.

In [105]:
X.columns

Index(['decision_text', 'key_rate', 'key_rate_prev', 'inflation',
       'inflation_lag_2', 'inflation_gap', 'usd_rate', 'text_len_tokens',
       'hike_words_ratio', 'cut_words_ratio', 'hold_words_ratio',
       'hike_minus_cut_ratio'],
      dtype='object')

In [106]:
# делаем копию дф
temp = df.copy()
temp = temp.drop(['date', 'title_clean', 'text_clean', 'text_tokens', 'url'], axis=1)
# выделяем текущие колонки
col = temp.columns

# выводим корреляцию между целеой переменной и данными
corr_with_target = temp.corr()['decision_future'].drop('decision_future')
corr_sorted = corr_with_target.reindex(
    corr_with_target.abs().sort_values(ascending=False).index
)

print(corr_sorted)


decision_text           0.721580
usd_rate                0.440719
usd_lag_1               0.426315
usd_lag_2               0.417862
usd_lag_3               0.412966
hold_words_ratio       -0.372234
cut_words_ratio        -0.364135
text_unique_tokens      0.346427
cut_words_count        -0.343972
text_len_chars          0.311428
text_len_tokens         0.311235
hike_minus_cut_ratio    0.302916
hike_words_count        0.297451
inflation_lag_1        -0.239152
inflation_lag_3        -0.231923
inflation_lag_2        -0.225208
inflation              -0.193069
inflation_gap          -0.193069
hold_words_count        0.176378
text_unique_ratio      -0.171949
key_rate_prev          -0.107983
key_rate               -0.107366
hike_words_ratio        0.083993
key_rate_change         0.010099
Name: decision_future, dtype: float64


inflation_lag_1 больше коррелирует чем inflation_lag_2, поэтому его стоит поменять, inflation можно убрать, так как есть лаговая инфляция, inflation_gap показал слабое влияние и его тоже стоит убрать, <br>
text_len_chars и text_len_tokens практически одинаковы, можно убрать один <br>
hold_words_count, text_unique_ratio, key_rate_prev, hike_words_ratio, key_rate_change - стоит убрать
key_rate - оставляем так как это важная метрика

In [123]:
X = df[['decision_text', 'key_rate', 'inflation_lag_1', 'usd_rate', 'usd_lag_1', 
        'usd_lag_2', 'usd_lag_3','text_len_tokens', 'text_unique_tokens', 'hike_words_count',
        'cut_words_count', 'cut_words_ratio', 'hold_words_ratio', 'hike_minus_cut_ratio']]
y = df['decision_future']

In [111]:
# разделяем датасет на трейн и тест
Xtrain, Xtest, ytrain, ytest = train_test_split(X, y, test_size=0.25, random_state=42)

# выделяем колонки для стандартизации
stand_col = ['key_rate', 'inflation_lag_1', 'usd_rate', 'usd_lag_1', 'usd_lag_2', 'usd_lag_3',
              'text_len_tokens', 'text_unique_tokens', 'hike_words_count', 'cut_words_count']

# делаем объект масштабирования
scaler = StandardScaler()

# делаем копии, чтобы не трогать текущие данные
Xtrain_scaled = Xtrain.copy()
Xtest_scaled = Xtest.copy()
# считаем параметры стандартизации на трейне и стандартизируем признаки
Xtrain_scaled[stand_col] = scaler.fit_transform(Xtrain[stand_col])
# стандартизируем признаки для тест
Xtest_scaled[stand_col] = scaler.transform(Xtest[stand_col])

In [115]:
from sklearn.model_selection import GridSearchCV

knn = KNeighborsClassifier()

param_grid_knn = {
    'n_neighbors': [3, 5, 7, 9, 11, 12, 13, 14, 15, 16, 17],
    'weights': ['uniform', 'distance']
}

grid_knn = GridSearchCV(
    estimator=knn,
    param_grid=param_grid_knn,
    scoring='f1_macro',
    cv=5
)

grid_knn.fit(Xtrain_scaled, ytrain)

print("KNN лучшие параметры:", grid_knn.best_params_)
print("KNN лучшая F1:", grid_knn.best_score_)

# оценка на тесте
best_knn = grid_knn.best_estimator_
y_pred_knn = best_knn.predict(Xtest_scaled)

acc_knn_gs = accuracy_score(ytest, y_pred_knn)
f1_knn_gs = f1_score(ytest, y_pred_knn, average='macro')

print("KNN test Accuracy:", acc_knn_gs)
print("KNN test F1:", f1_knn_gs)

KNN лучшие параметры: {'n_neighbors': 3, 'weights': 'uniform'}
KNN лучшая F1: 0.7219576719576719
KNN test Accuracy: 0.6428571428571429
KNN test F1: 0.6560846560846562


Как видно, качество модели после смены признаков повысилось. Однако можно улучшить качество модели с помощью смены нормализации признаков.

In [124]:
# разделяем датасет на трейн и тест
Xtrain, Xtest, ytrain, ytest = train_test_split(X, y, test_size=0.25, random_state=42)

# выделяем колонки для стандартизации
stand_col = ['key_rate', 'inflation_lag_1', 'usd_rate', 'usd_lag_1', 'usd_lag_2', 'usd_lag_3',
              'text_len_tokens', 'text_unique_tokens', 'hike_words_count', 'cut_words_count']

# делаем объект масштабирования
scaler = MinMaxScaler()

# делаем копии, чтобы не трогать текущие данные
Xtrain_scaled = Xtrain.copy()
Xtest_scaled = Xtest.copy()
# считаем параметры стандартизации на трейне и стандартизируем признаки
Xtrain_scaled[stand_col] = scaler.fit_transform(Xtrain[stand_col])
# стандартизируем признаки для тест
Xtest_scaled[stand_col] = scaler.transform(Xtest[stand_col])

In [125]:
from sklearn.model_selection import GridSearchCV

knn = KNeighborsClassifier()

param_grid_knn = {
    'n_neighbors': [3, 5, 7, 9, 11, 12, 13, 14, 15, 16, 17],
    'weights': ['uniform', 'distance']
}

grid_knn = GridSearchCV(
    estimator=knn,
    param_grid=param_grid_knn,
    scoring='f1_macro',
    cv=5
)

grid_knn.fit(Xtrain_scaled, ytrain)

print("KNN лучшие параметры:", grid_knn.best_params_)
print("KNN лучшая F1:", grid_knn.best_score_)

# оценка на тесте
best_knn = grid_knn.best_estimator_
y_pred_knn = best_knn.predict(Xtest_scaled)

acc_knn_gs_best = accuracy_score(ytest, y_pred_knn)
f1_knn_gs_best = f1_score(ytest, y_pred_knn, average='macro')

print("KNN test Accuracy:", acc_knn_gs_best)
print("KNN test F1:", f1_knn_gs_best)

KNN лучшие параметры: {'n_neighbors': 3, 'weights': 'uniform'}
KNN лучшая F1: 0.7078306878306877
KNN test Accuracy: 0.7857142857142857
KNN test F1: 0.7857142857142857


Таким образом качество модели получилось увеличить до 75 для f1 и Accuracy

# Модели LogisticRegression и LinearRegression

In [85]:
# разделяем датасет на трейн и тест
Xtrain, Xtest, ytrain, ytest = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)

In [54]:
# выделяем колонки для стандартизации
stand_col = ['key_rate', 'key_rate_prev', 'inflation', 'inflation_lag_2', 
             'inflation_gap', 'usd_rate', 'text_len_tokens']

# делаем объект масштабирования
scaler = StandardScaler()

# делаем копии, чтобы не трогать текущие данные
Xtrain_scaled = Xtrain.copy()
Xtest_scaled = Xtest.copy()
# считаем параметры стандартизации на трейне и стандартизируем признаки
Xtrain_scaled[stand_col] = scaler.fit_transform(Xtrain[stand_col])
# стандартизируем признаки для тест
Xtest_scaled[stand_col] = scaler.transform(Xtest[stand_col])

In [ ]:
# Создание модели с фиксированными параметрами
model = LogisticRegression()

# Обучение модели
model.fit(Xtrain_scaled, ytrain)

# Оценка модели на тесте
y_pred = model.predict(Xtest_scaled)

acc_lr = accuracy_score(ytest, y_pred)
f1_lr = f1_score(ytest, y_pred, average='macro')

print("Accuracy:", acc_lr)
print("f1:", f1_lr)

Accuracy: 0.6428571428571429
f1: 0.6428571428571429


Общая точность Accuracy = 0.64 показывает, что 70% всех предсказаний модель делает правильно.

F1 метрика = 0.66 — среднее между точностью положительных предсказаний и долей найденных истинных положительных.

То, что Accuracy и F1 примерно совпадают, говорить о том, что модель различает классы при предсказании.

In [56]:
print(y_pred)
print(ytest)

[-1.  0. -1.  1. -1.  0.  0.  1.  0.  0.  1.  1.  1.  1.]
27   -1.0
48    0.0
5    -1.0
1     0.0
8    -1.0
31    0.0
47    0.0
12    0.0
45    1.0
25   -1.0
2     0.0
18    1.0
20    1.0
36    1.0
Name: decision_future, dtype: float64


In [57]:
X2 = df[['decision_text', 'key_rate_prev', 'inflation','inflation_lag_2', 
             'inflation_gap', 'usd_rate', 'text_len_tokens', 'hike_words_ratio', 'cut_words_ratio',
             'hold_words_ratio', 'hike_minus_cut_ratio']]
y2 = df['key_rate']

In [58]:
# Разделение данных
X2_train, X2_test, y2_train, y2_test = train_test_split(
    X2, y2, test_size=0.20, random_state=42)

    # Масштабирование
scaler = StandardScaler()
X2_train_scaled = scaler.fit_transform(X2_train)
X2_test_scaled = scaler.transform(X2_test)

    # Создание модели с фиксированными параметрами
model = LinearRegression()

                # Обучение модели
model.fit(X2_train_scaled, y2_train)

                # Оценка модели на тесте

y2_pred = model.predict(X2_test_scaled)

In [59]:
mape = MAPE(y2_test, y2_pred)*100
# Вывод результатов
print(f"MAPE: {mape:.1f}%")

MAPE: 1.1%


в среднем предсказанные значения отклоняются от истинных на ~1.1%. То есть,ошибка около 15–16% от реального.

In [60]:
print(y2_pred)
print(y2_test)

[ 7.39496725 20.74285219 20.60111146  4.09482935 18.20128583  7.27988804
  6.47271387 17.95586187  7.80400649  7.48966202  4.21706788]
19     7.50
49    21.00
48    21.00
12     4.25
44    18.00
5      7.25
17     6.50
52    18.00
3      7.75
32     7.50
13     4.25
Name: key_rate, dtype: float64


## Выводы работы

In [134]:
acc = {'SVC': acc_svc, 'SVC + GridSearch': acc_svc_gs,'KNN': acc_knn, 'KNN + MinMax + смена признаков': acc_knn_gs_best, 'Logistic Regression': acc_lr}
f1 = {'SVC': f1_svc, 'SVC + GridSearch': f1_svc_gs, 'KNN': f1_knn, 'KNN + MinMax + смена признаков': f1_knn_gs_best, 'Logistic Regression': f1_lr}

df_metrics = pd.DataFrame({
    'Accuracy': acc,
    'F1': f1
})

print('Метрики качества моделей')
df_metrics

Метрики качества моделей


,Accuracy,F1
SVC,0.500000,0.511111
SVC + GridSearch,0.714286,0.694444
KNN,0.642857,0.657885
KNN + MinMax + смена признаков,0.785714,0.785714
Logistic Regression,0.642857,0.642857


1. **SVC** работает плохо, предсказывает только половину.
Это объясняется высокой чувствительностью SVC к масштабам признаков и параметрам. 
Несмотря на стандартизацию числовых признаков, SVC плохо работает с небольшими объёмами выборки, слабой линейной разделимостью классов, шумными текстовыми фичами.
Скорее всего на 54 строках модель не может построить устойчивую разделяющую поверхность.

2. **KNN** показывает лучшую метрику
KNN хорошо работает на небольших датасетах так как опирается на ближайшие примеры.
Модель также невосприимчив к линейности.
Но KNN склонна к переобучению, поэтому стоит проверить на кросс-валидации

3. **Logistic Regression** показывает средний результат
Логистическая регрессия ищет линейную границу между классами, но решение ЦБ — нелинейная функция от макроэкономики, текстовые признаки тоже не образуют линейно разделяемое пространство.
Используются недостаточно данных о тексте, что упрощает представление текста поэтому линейная модель не получает достаточного количества сигналов.

**Что можно добавить:**
- увеличить датасет за счет пересмотра решений, так как большая часть пресс-релизов отсеилась по данному признаку
- добавить TF-IDF признаки
- применить кросс-валидацию
- настроить гиперпараметры в моделях

Также как видно из таблицы за счет смены признаков - их пересмотр через корреляционную зависимость с целевой переменной и смены стандартизации признаков качество модели KNN увеличилось. <br>
За счет GridSearch увеличилось качество модели SVM